In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===== RUTA BASE =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
base_dir.mkdir(parents=True, exist_ok=True)

# ===== ARCHIVOS DE ENTRADA =====
pc_file = base_dir / "pc componentes scrapping limpio.csv"
kg_file = base_dir / "dataset amazon electronics 2025 limpio.csv"

# ===== ARCHIVO DE SALIDA =====
macro_file = base_dir / "macrodataset_gaming.csv"

# ===== CARGA =====
pc = pd.read_csv(pc_file)
kg = pd.read_csv(kg_file)

# Normalizar nombres de columnas
pc.columns = pc.columns.str.strip().str.lower()
kg.columns = kg.columns.str.strip().str.lower()

# ===== LIMPIEZA BÁSICA DE TEXTO =====
def clean_text_series(s):
    return s.astype("string").str.strip()

# ===== ESTANDARIZAR PC COMPONENTES =====
pc_macro = pd.DataFrame({
    "source": "pccomponentes",
    "product_name": pc["nombre"] if "nombre" in pc.columns else np.nan,
    "brand": pc["marca"] if "marca" in pc.columns else np.nan,
    "category": pc["subcategoria"] if "subcategoria" in pc.columns else np.nan,
    "subcategory": pc["subcategoria"] if "subcategoria" in pc.columns else np.nan,
    "price": pc["precio_actual"] if "precio_actual" in pc.columns else np.nan,
    "price_original": pc["precio_original"] if "precio_original" in pc.columns else np.nan,
    "discount_pct": pc["descuento_pct"] if "descuento_pct" in pc.columns else np.nan,
    "rating": pc["rating"] if "rating" in pc.columns else np.nan,
    "review_count": pc["num_opiniones"] if "num_opiniones" in pc.columns else np.nan,
    "popularity": pc["num_opiniones"] if "num_opiniones" in pc.columns else np.nan,
    "product_url": pc["product_url"] if "product_url" in pc.columns else np.nan,
    "seller": pc["seller_raw"] if "seller_raw" in pc.columns else np.nan,
    "promo_tag": pc["promo_raw"] if "promo_raw" in pc.columns else np.nan,
    "sku": pc["sku"] if "sku" in pc.columns else np.nan,
})

# ===== ESTANDARIZAR KAGGLE AMAZON =====
# Ajusta aquí si tu dataset usa otros nombres
kg_macro = pd.DataFrame({
    "source": "kaggle_amazon",
    "product_name": kg["product_name"] if "product_name" in kg.columns else kg["name"] if "name" in kg.columns else np.nan,
    "brand": kg["brand"] if "brand" in kg.columns else np.nan,
    "category": kg["category"] if "category" in kg.columns else np.nan,
    "subcategory": kg["subcategory"] if "subcategory" in kg.columns else np.nan,
    "price": kg["price"] if "price" in kg.columns else np.nan,
    "price_original": kg["price_original"] if "price_original" in kg.columns else np.nan,
    "discount_pct": kg["discount_pct"] if "discount_pct" in kg.columns else np.nan,
    "rating": kg["rating"] if "rating" in kg.columns else np.nan,
    "review_count": kg["review_count"] if "review_count" in kg.columns else np.nan,
    "popularity": kg["sales"] if "sales" in kg.columns else kg["popularity"] if "popularity" in kg.columns else np.nan,
    "product_url": kg["product_url"] if "product_url" in kg.columns else np.nan,
    "seller": kg["seller"] if "seller" in kg.columns else np.nan,
    "promo_tag": kg["promo_tag"] if "promo_tag" in kg.columns else np.nan,
    "sku": kg["sku"] if "sku" in kg.columns else np.nan,
})

# ===== CONCATENAR =====
macro = pd.concat([pc_macro, kg_macro], ignore_index=True)

# ===== NORMALIZAR TIPO DE DATOS =====
for col in ["price", "price_original", "discount_pct", "rating", "review_count", "popularity"]:
    macro[col] = pd.to_numeric(macro[col], errors="coerce")

for col in ["product_name", "brand", "category", "subcategory", "product_url", "seller", "promo_tag", "source"]:
    macro[col] = clean_text_series(macro[col])

# ===== LIMPIEZA DE NULOS BÁSICOS =====
macro = macro.dropna(subset=["product_name"], how="all").copy()
macro = macro.dropna(subset=["price"], how="all").copy()

# ===== ELIMINAR DUPLICADOS RAZONABLES =====
macro = macro.drop_duplicates(
    subset=["source", "product_name", "brand", "price"],
    keep="first"
).copy()

# ===== ORDENAR COLUMNAS =====
ordered_cols = [
    "source", "product_name", "brand", "category", "subcategory",
    "price", "price_original", "discount_pct", "rating",
    "review_count", "popularity", "seller", "promo_tag",
    "product_url", "sku"
]
macro = macro[[c for c in ordered_cols if c in macro.columns]].copy()

# ===== GUARDAR =====
macro.to_csv(macro_file, index=False, encoding="utf-8-sig")

print(f"Macrodataset guardado en: {macro_file}")
print(f"Filas totales: {len(macro)}")
print(macro.head())

Macrodataset guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping\macrodataset_gaming.csv
Filas totales: 231
          source                                       product_name  brand  \
0  pccomponentes  PcCom Ready AMD Ryzen 7 5800X / 32GB / 1TB SSD...  PcCom   
1  pccomponentes  PcCom Imperial AMD Ryzen 7 5800X / 32GB / 2TB ...  PcCom   
2  pccomponentes  PcCom Imperial AMD Ryzen 7 5800X / 32GB / 2TB ...  PcCom   
3  pccomponentes  PcCom Imperial AMD Ryzen 7 5700X / 32GB / 1TB ...  PcCom   
4  pccomponentes  PcCom Ready AMD Ryzen 7 5800X / 32GB / 1TB SSD...  PcCom   

  category subcategory    price  price_original  discount_pct  rating  \
0       PC          PC  12990.0             NaN           NaN     NaN   
1       PC          PC  14990.0             NaN           NaN     NaN   
2       PC          PC  15790.0             NaN           NaN     NaN   
3       PC          PC  14490.0             NaN           NaN  

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
import re

# ===== RUTA BASE =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
base_dir.mkdir(parents=True, exist_ok=True)

# ===== ARCHIVOS DE ENTRADA =====
pc_file = base_dir / "pc componentes scrapping limpio.csv"
kg_file = base_dir / "dataset amazon electronics 2025 limpio.csv"

# ===== ARCHIVO DE SALIDA =====
macro_file = base_dir / "Dataset Maestro.csv"

# ===== CARGA =====
pc = pd.read_csv(pc_file)
kg = pd.read_csv(kg_file)

# Normalizar nombres de columnas
pc.columns = pc.columns.str.strip().str.lower()
kg.columns = kg.columns.str.strip().str.lower()

# ===== LIMPIEZA BÁSICA DE TEXTO =====
def clean_text_series(s):
    return s.astype("string").str.strip()

# ===== LIMPIEZA / NORMALIZACIÓN DE URLS =====
def normalize_url(url):
    if pd.isna(url) or str(url).strip() == "":
        return pd.NA

    url = str(url).strip()

    md_match = re.search(r"\((https?://[^)]+)\)", url)
    if md_match:
        url = md_match.group(1)

    url = url.strip("[]")

    if url.startswith("www."):
        url = "https://" + url
    elif not url.startswith(("http://", "https://")):
        url = "https://" + url.lstrip("/")

    parsed = urlparse(url)

    scheme = "https"
    netloc = parsed.netloc.lower().replace("www.", "")
    path = parsed.path.replace("//", "/").strip()

    while "//" in path:
        path = path.replace("//", "/")

    if path != "/":
        path = path.rstrip("/")

    query_params = parse_qsl(parsed.query, keep_blank_values=True)
    query_params = [(k, v) for k, v in query_params if not k.lower().startswith("utm_")]
    query = urlencode(sorted(query_params))

    return urlunparse((scheme, netloc, path, "", query, ""))

# ===== ESTANDARIZAR PC COMPONENTES =====
pc_macro = pd.DataFrame({
    "source": "pccomponentes",
    "product_name": pc["nombre"] if "nombre" in pc.columns else np.nan,
    "brand": pc["marca"] if "marca" in pc.columns else np.nan,
    "category": pc["subcategoria"] if "subcategoria" in pc.columns else np.nan,
    "subcategory": pc["subcategoria"] if "subcategoria" in pc.columns else np.nan,
    "price": pc["precio_actual"] if "precio_actual" in pc.columns else np.nan,
    "price_original": pc["precio_original"] if "precio_original" in pc.columns else np.nan,
    "discount_pct": pc["descuento_pct"] if "descuento_pct" in pc.columns else np.nan,
    "rating": pc["rating"] if "rating" in pc.columns else np.nan,
    "review_count": pc["num_opiniones"] if "num_opiniones" in pc.columns else np.nan,
    "popularity": pc["num_opiniones"] if "num_opiniones" in pc.columns else np.nan,
    "product_url": pc["product_url"] if "product_url" in pc.columns else np.nan,
    "seller": pc["seller_raw"] if "seller_raw" in pc.columns else np.nan,
    "promo_tag": pc["promo_raw"] if "promo_raw" in pc.columns else np.nan,
    "sku": pc["sku"] if "sku" in pc.columns else np.nan,
})

# ===== ESTANDARIZAR KAGGLE AMAZON =====
kg_macro = pd.DataFrame({
    "source": "kaggle_amazon",
    "product_name": kg["product_name"] if "product_name" in kg.columns else kg["name"] if "name" in kg.columns else np.nan,
    "brand": kg["brand"] if "brand" in kg.columns else np.nan,
    "category": kg["category"] if "category" in kg.columns else np.nan,
    "subcategory": kg["subcategory"] if "subcategory" in kg.columns else np.nan,
    "price": kg["price"] if "price" in kg.columns else np.nan,
    "price_original": kg["price_original"] if "price_original" in kg.columns else np.nan,
    "discount_pct": kg["discount_pct"] if "discount_pct" in kg.columns else np.nan,
    "rating": kg["rating"] if "rating" in kg.columns else np.nan,
    "review_count": kg["review_count"] if "review_count" in kg.columns else np.nan,
    "popularity": kg["sales"] if "sales" in kg.columns else kg["popularity"] if "popularity" in kg.columns else np.nan,
    "product_url": kg["product_url"] if "product_url" in kg.columns else np.nan,
    "seller": kg["seller"] if "seller" in kg.columns else np.nan,
    "promo_tag": kg["promo_tag"] if "promo_tag" in kg.columns else np.nan,
    "sku": kg["sku"] if "sku" in kg.columns else np.nan,
})

# ===== CONCATENAR =====
macro = pd.concat([pc_macro, kg_macro], ignore_index=True)

# ===== NORMALIZAR TIPO DE DATOS =====
for col in ["price", "price_original", "discount_pct", "rating", "review_count", "popularity"]:
    macro[col] = pd.to_numeric(macro[col], errors="coerce")

for col in ["product_name", "brand", "category", "subcategory", "product_url", "seller", "promo_tag", "source", "sku"]:
    macro[col] = clean_text_series(macro[col])

# ===== LIMPIEZA DE URLS =====
macro["product_url_clean"] = macro["product_url"].apply(normalize_url)
macro["url_mal_formateada"] = macro["product_url_clean"].isna()
macro["product_url"] = macro["product_url_clean"]
macro = macro.drop(columns=["product_url_clean"])

# ===== LIMPIEZA DE NULOS BÁSICOS =====
macro = macro.dropna(subset=["product_name"], how="all").copy()
macro = macro.dropna(subset=["price"], how="all").copy()

# ===== NORMALIZAR CAMPOS PARA DUPLICADOS =====
macro["product_name_norm"] = (
    macro["product_name"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

macro["brand_norm"] = (
    macro["brand"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

macro["subcategory_norm"] = (
    macro["subcategory"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# ===== DUPLICADOS EVIDENTES =====
macro["duplicado_evidente"] = macro.duplicated(
    subset=["source", "product_name_norm", "brand_norm", "price"],
    keep=False
)

macro["duplicado_url"] = macro.duplicated(
    subset=["source", "product_url"],
    keep=False
)

macro = macro.drop_duplicates(
    subset=["source", "product_name_norm", "brand_norm", "price"],
    keep="first"
).copy()

# ===== LIMPIEZA FINAL DE COLUMNAS AUXILIARES =====
macro = macro.drop(
    columns=[
        "product_name_norm",
        "brand_norm",
        "subcategory_norm",
        "duplicado_evidente"
    ],
    errors="ignore"
)

# ===== ORDENAR COLUMNAS =====
ordered_cols = [
    "source", "product_name", "brand", "category", "subcategory",
    "price", "price_original", "discount_pct", "rating",
    "review_count", "popularity", "seller", "promo_tag",
    "product_url", "sku", "url_mal_formateada", "duplicado_url"
]
macro = macro[[c for c in ordered_cols if c in macro.columns]].copy()

# ===== GUARDAR =====
macro.to_csv(macro_file, index=False, encoding="utf-8-sig")

print(f"Macrodataset guardado en: {macro_file}")
print(f"Filas totales: {len(macro)}")
print(macro.head())

Macrodataset guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping\Dataset Maestro.csv
Filas totales: 231
          source                                       product_name  brand  \
0  pccomponentes  PcCom Ready AMD Ryzen 7 5800X / 32GB / 1TB SSD...  PcCom   
1  pccomponentes  PcCom Imperial AMD Ryzen 7 5800X / 32GB / 2TB ...  PcCom   
2  pccomponentes  PcCom Imperial AMD Ryzen 7 5800X / 32GB / 2TB ...  PcCom   
3  pccomponentes  PcCom Imperial AMD Ryzen 7 5700X / 32GB / 1TB ...  PcCom   
4  pccomponentes  PcCom Ready AMD Ryzen 7 5800X / 32GB / 1TB SSD...  PcCom   

  category subcategory    price  price_original  discount_pct  rating  \
0       PC          PC  12990.0             NaN           NaN     NaN   
1       PC          PC  14990.0             NaN           NaN     NaN   
2       PC          PC  15790.0             NaN           NaN     NaN   
3       PC          PC  14490.0             NaN           NaN     N